In [1]:
import pandas as pd
import re

# Load dataset
df = pd.read_csv("/kaggle/input/sample-emails/sample_emails.csv")

In [2]:
df.head()

,id,sender,subject,body,priority,label
0,1,hr@company.com,Offer Letter - Please Sign,"Dear Candidate, Congratulations! Please sign a...",high,important
1,2,noreply@shopping.com,Big Billion Sale is Live,Flat 70% OFF on electronics. Limited period of...,low,promotion
2,3,alerts@mybank.com,Unusual Login Attempt,We detected a login attempt to your account fr...,high,security
3,4,newsletter@blog.com,Weekly Tech Newsletter,"In this week's edition, learn about AI agents,...",low,newsletter
4,5,friend123@gmail.com,Coffee this weekend?,"Hey, long time no see! Are you free this weeke...",medium,personal


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        8 non-null      int64 
 1   sender    8 non-null      object
 2   subject   8 non-null      object
 3   body      8 non-null      object
 4   priority  8 non-null      object
 5   label     8 non-null      object
dtypes: int64(1), object(5)
memory usage: 516.0+ bytes


In [22]:
df.describe(include="all")

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,id,sender,subject,body,priority,label,combined,clean_text,clean_no_stop
count,8.00000,8,8,8,8,8,8,8,8
unique,NaN,8,8,8,3,7,8,8,8
top,NaN,hr@company.com,offer letter please sign,dear candidate congratulations please sign and upload your offer letter by today evening.,high,work,offer letter please sign dear candidate congratulations please sign and upload your offer letter by today evening.,dear candidate congratulations please sign and upload your offer letter by today evening,dear candidate congratulations please sign upload offer letter today evening
freq,NaN,1,1,1,4,2,1,1,1
mean,4.50000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,2.44949,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,2.75000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,4.50000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,6.25000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df["label"].value_counts()

label
work          2
important     1
promotion     1
security      1
newsletter    1
personal      1
spam          1
Name: count, dtype: int64

In [5]:
print(df.columns)

Index(['id', 'sender', 'subject', 'body', 'priority', 'label'], dtype='object')


In [6]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = x.lower().strip()
    x = re.sub(r'\s+', ' ', x)  # collapse spaces
    x = re.sub(r'[^\w\s@.?]', '', x)  # keep words, email chars, ., ?
    return x

In [7]:
text_cols = ['sender', 'subject', 'body', 'priority', 'label']

for col in text_cols:
    df[col] = df[col].apply(clean_text)

In [8]:
pd.set_option('display.max_colwidth', None)
df

,id,sender,subject,body,priority,label
0,1,hr@company.com,offer letter please sign,dear candidate congratulations please sign and upload your offer letter by today evening.,high,important
1,2,noreply@shopping.com,big billion sale is live,flat 70 off on electronics. limited period offer. visit our store now.,low,promotion
2,3,alerts@mybank.com,unusual login attempt,we detected a login attempt to your account from a new device. if this was not you please reset your password.,high,security
3,4,newsletter@blog.com,weekly tech newsletter,in this weeks edition learn about ai agents langgraph and more.,low,newsletter
4,5,friend123@gmail.com,coffee this weekend?,hey long time no see are you free this weekend for coffee?,medium,personal
5,6,spam@strangedomain.ru,you won 1000000,click here to claim your lottery now. do not share this with anyone.,high,spam
6,7,manager@company.com,project update required,please share the status update of your project before 5 pm today.,high,work
7,8,training@lms.com,reminder complete mandatory training,your mandatory security awareness training is due tomorrow.,medium,work


In [9]:
df['combined'] = (df['subject'] + " " + df['body']).str.strip()
df['combined'] = df['combined'].apply(clean_text)

df[['id', 'sender', 'combined']]

,id,sender,combined
0,1,hr@company.com,offer letter please sign dear candidate congratulations please sign and upload your offer letter by today evening.
1,2,noreply@shopping.com,big billion sale is live flat 70 off on electronics. limited period offer. visit our store now.
2,3,alerts@mybank.com,unusual login attempt we detected a login attempt to your account from a new device. if this was not you please reset your password.
3,4,newsletter@blog.com,weekly tech newsletter in this weeks edition learn about ai agents langgraph and more.
4,5,friend123@gmail.com,coffee this weekend? hey long time no see are you free this weekend for coffee?
5,6,spam@strangedomain.ru,you won 1000000 click here to claim your lottery now. do not share this with anyone.
6,7,manager@company.com,project update required please share the status update of your project before 5 pm today.
7,8,training@lms.com,reminder complete mandatory training your mandatory security awareness training is due tomorrow.


In [10]:
df[df['combined'].str.contains("offer", case=False)]

,id,sender,subject,body,priority,label,combined
0,1,hr@company.com,offer letter please sign,dear candidate congratulations please sign and upload your offer letter by today evening.,high,important,offer letter please sign dear candidate congratulations please sign and upload your offer letter by today evening.
1,2,noreply@shopping.com,big billion sale is live,flat 70 off on electronics. limited period offer. visit our store now.,low,promotion,big billion sale is live flat 70 off on electronics. limited period offer. visit our store now.


In [11]:
df[df['priority'] == "high"]

,id,sender,subject,body,priority,label,combined
0,1,hr@company.com,offer letter please sign,dear candidate congratulations please sign and upload your offer letter by today evening.,high,important,offer letter please sign dear candidate congratulations please sign and upload your offer letter by today evening.
2,3,alerts@mybank.com,unusual login attempt,we detected a login attempt to your account from a new device. if this was not you please reset your password.,high,security,unusual login attempt we detected a login attempt to your account from a new device. if this was not you please reset your password.
5,6,spam@strangedomain.ru,you won 1000000,click here to claim your lottery now. do not share this with anyone.,high,spam,you won 1000000 click here to claim your lottery now. do not share this with anyone.
6,7,manager@company.com,project update required,please share the status update of your project before 5 pm today.,high,work,project update required please share the status update of your project before 5 pm today.


In [12]:
df[df['label'].isin(["important", "security"])]

,id,sender,subject,body,priority,label,combined
0,1,hr@company.com,offer letter please sign,dear candidate congratulations please sign and upload your offer letter by today evening.,high,important,offer letter please sign dear candidate congratulations please sign and upload your offer letter by today evening.
2,3,alerts@mybank.com,unusual login attempt,we detected a login attempt to your account from a new device. if this was not you please reset your password.,high,security,unusual login attempt we detected a login attempt to your account from a new device. if this was not you please reset your password.


In [13]:
df[df['combined'].str.contains("login", case=False)]

,id,sender,subject,body,priority,label,combined
2,3,alerts@mybank.com,unusual login attempt,we detected a login attempt to your account from a new device. if this was not you please reset your password.,high,security,unusual login attempt we detected a login attempt to your account from a new device. if this was not you please reset your password.


In [14]:
df[df['sender'] != "newsletter@blog.com"]

,id,sender,subject,body,priority,label,combined
0,1,hr@company.com,offer letter please sign,dear candidate congratulations please sign and upload your offer letter by today evening.,high,important,offer letter please sign dear candidate congratulations please sign and upload your offer letter by today evening.
1,2,noreply@shopping.com,big billion sale is live,flat 70 off on electronics. limited period offer. visit our store now.,low,promotion,big billion sale is live flat 70 off on electronics. limited period offer. visit our store now.
2,3,alerts@mybank.com,unusual login attempt,we detected a login attempt to your account from a new device. if this was not you please reset your password.,high,security,unusual login attempt we detected a login attempt to your account from a new device. if this was not you please reset your password.
4,5,friend123@gmail.com,coffee this weekend?,hey long time no see are you free this weekend for coffee?,medium,personal,coffee this weekend? hey long time no see are you free this weekend for coffee?
5,6,spam@strangedomain.ru,you won 1000000,click here to claim your lottery now. do not share this with anyone.,high,spam,you won 1000000 click here to claim your lottery now. do not share this with anyone.
6,7,manager@company.com,project update required,please share the status update of your project before 5 pm today.,high,work,project update required please share the status update of your project before 5 pm today.
7,8,training@lms.com,reminder complete mandatory training,your mandatory security awareness training is due tomorrow.,medium,work,reminder complete mandatory training your mandatory security awareness training is due tomorrow.


In [15]:
df.shape

(8, 7)

In [16]:
df.isnull().sum()

id          0
sender      0
subject     0
body        0
priority    0
label       0
combined    0
dtype: int64

In [17]:
df['clean_text'] = df['body'].str.lower().str.replace(r'[^a-zA-Z\s]', '', regex=True)
df[['body', 'clean_text']].head()

,body,clean_text
0,dear candidate congratulations please sign and upload your offer letter by today evening.,dear candidate congratulations please sign and upload your offer letter by today evening
1,flat 70 off on electronics. limited period offer. visit our store now.,flat off on electronics limited period offer visit our store now
2,we detected a login attempt to your account from a new device. if this was not you please reset your password.,we detected a login attempt to your account from a new device if this was not you please reset your password
3,in this weeks edition learn about ai agents langgraph and more.,in this weeks edition learn about ai agents langgraph and more
4,hey long time no see are you free this weekend for coffee?,hey long time no see are you free this weekend for coffee


In [18]:
from collections import Counter

all_words = " ".join(df['clean_text']).split()
word_freq = Counter(all_words)

word_freq.most_common(3)

[('your', 6), ('this', 4), ('please', 3)]

In [19]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

df['clean_no_stop'] = df['clean_text'].apply(
    lambda x: " ".join([word for word in x.split() if word not in stop_words])
)

df[['clean_text', 'clean_no_stop']].head()

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,clean_text,clean_no_stop
0,dear candidate congratulations please sign and upload your offer letter by today evening,dear candidate congratulations please sign upload offer letter today evening
1,flat off on electronics limited period offer visit our store now,flat electronics limited period offer visit store
2,we detected a login attempt to your account from a new device if this was not you please reset your password,detected login attempt account new device please reset password
3,in this weeks edition learn about ai agents langgraph and more,weeks edition learn ai agents langgraph
4,hey long time no see are you free this weekend for coffee,hey long time see free weekend coffee


In [ ]:
# Email: "Please approve the project status update urgently!"

In [23]:
#Without NLTK
["please","approve","the","project","status","update","urgently"]

#With NLTK Stopword Removal
["please","approve","project","status","update","urgently"]

['please', 'approve', 'project', 'status', 'update', 'urgently']

In [24]:
#Spam Detection Code
import re

SPAM_KEYWORDS = [
    "win", "winner", "lottery", "jackpot",
    "free", "offer", "limited time", "limited period",
    "discount", "sale", "clearance", "deal",
    "click here", "buy now", "urgent",
    "congratulations", "prize"
]

# Compile regex once
pattern = re.compile("|".join(re.escape(w) for w in SPAM_KEYWORDS), re.IGNORECASE)

def is_spam(text: str) -> int:
    if not isinstance(text, str):
        return 0
    return int(bool(pattern.search(text)))

In [25]:
# Ensure subject/body columns exist
if 'subject' not in df.columns:
    df['subject'] = ''

if 'body' not in df.columns:
    df['body'] = ''

# Combine subject and body for detection
df["text"] = df["subject"].fillna("").astype(str) + " " + df["body"].fillna("").astype(str)
df["text"] = df["text"].str.strip()

# Binary spam flag
df["is_spam_rule"] = df["text"].apply(is_spam)

print(df[["id", "subject", "is_spam_rule"]])

   id                               subject  is_spam_rule
0   1             offer letter  please sign             1
1   2              big billion sale is live             1
2   3                 unusual login attempt             0
3   4                weekly tech newsletter             0
4   5                  coffee this weekend?             1
5   6                       you won 1000000             1
6   7               project update required             0
7   8  reminder complete mandatory training             0


In [20]:
pd.DataFrame(word_freq.most_common(20), columns=["word", "count"])

,word,count
0,your,6
1,this,4
2,please,3
3,and,2
4,offer,2
5,today,2
6,now,2
7,a,2
8,to,2
9,not,2
